# Ordered Logistic Regression Results for Adoption Predictors – Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will explore record sets, fields, and columns using their `@id` fields, and perform some basic ETL and EDA tasks.

### Dataset Source

The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install required library (uncomment if running interactively)
!pip install mlcroissant

## 1. Data Loading

First, we'll load the dataset metadata and explore its structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Let's review the available record sets and their fields, using `@id` as the reference for each entity. This gives us insight into the structure and which pieces we can explore further.

In [ ]:
# List all available record sets by @id
print("Record Sets (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '(no name)')}")
    # List fields in this record set
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in fields:
            # f can be either a dict or a string (@id)
            if isinstance(f, dict):
                print(f"    - @id: {f.get('@id', str(f))}  | name: {f.get('name', '(no name)')}")
            else:
                print(f"    - @id: {f}")
    print("")

if not record_sets:
    print("No record sets listed in metadata. We'll try to explore using the records API.")

## 3. Data Extraction

Let's extract data from the available record sets. For each, we'll load the records into a pandas DataFrame for inspection. Note how we reference the record set by its `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
if not record_set_ids:
    print("No record set ids found. The dataset may define only a single logical record set or define them dynamically.\n")
else:
    print("Record set @ids:")
    for rset in record_set_ids:
        print(f"  - {rset}")

# For demo, select the first record set if exists
dataframes = {}
for rset_id in record_set_ids:
    print(f"\nLoading records for record set @id: {rset_id}")
    try:
        records = list(dataset.records(record_set=rset_id))
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f" - Loaded {len(df)} records with columns: {list(df.columns)}")
    except Exception as e:
        print(f" - Could not load records: {e}")

# Show a sample from the first record set, if available
if dataframes:
    first_rs = record_set_ids[0]
    print(f"\nExample rows for record set: {first_rs}")
    display(dataframes[first_rs].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Now, let's perform some EDA. We'll pick a numeric field using its `@id`, filter records, normalize data, and optionally group by a categorical attribute. Please change variable names to suit your schema as needed.

In [ ]:
import numpy as np

# Choose the record set you want to analyze:
if not dataframes:
    print("No data loaded for EDA.")
else:
    record_set_id = list(dataframes.keys())[0]  # Adjust if needed
    df = dataframes[record_set_id]
    print(f"Columns in record set {record_set_id}:")
    print(list(df.columns))

    # Try to pick a numeric field dynamically if available
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]   # Example: pick the first numeric field (override as needed)
        print(f"\nUsing numeric field: {numeric_field}")

        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.3f} (mean value): {len(filtered_df)} records")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try to group by a categorical field if available
        categorical_fields = df.select_dtypes(exclude=[np.number]).columns.tolist()
        group_field = None
        if categorical_fields:
            group_field = categorical_fields[0]  # Example: pick the first categorical field
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in chosen record set.")

## 5. Visualization

Let's visualize some data relationships. We'll plot the distribution of the numeric variable we just analyzed, and optionally a grouped bar/box plot if a group field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8, 4))
    if numeric_fields:
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    
    # Boxplot/grouped plot if group_field available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(y=df[numeric_field], x=df[group_field].astype(str))
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Accessed the FAIR^2 dataset via its Croissant schema and examined its metadata
- Explored the dataset structure, listing record sets and fields by `@id`
- Loaded available records into DataFrames
- Performed basic filtering, normalization, grouping, and visualizations

You can extend this workflow by selecting specific field `@id`s for deeper analysis, handling missing values, or exporting results for reporting or ML applications.